# Does Higher Internet Penetration Cause Lower Unemployment?
### DA4 Term Project — World Bank Panel Data Analysis
---
**Paper:** Atasoy, H. (2013). The Effects of Broadband Internet Expansion on Labor Market Outcomes. *ILR Review*, 66(2), 315–345.

**Data:**
- X: Internet users (% of population) — [World Bank](https://data.worldbank.org/indicator/IT.NET.USER.ZS)
- Y: Unemployment rate (% of labor force) — [World Bank](https://data.worldbank.org/indicator/SL.UEM.TOTL.ZS)

## 0. Install & Import Libraries

In [ ]:
# Run this once to install required packages
# !pip install pandas numpy matplotlib statsmodels linearmodels

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import statsmodels.formula.api as smf
from linearmodels.panel import PanelOLS
import warnings
warnings.filterwarnings('ignore')

print('Libraries loaded successfully!')

---
## 1. Load Data
**Instructions:**
1. Go to https://data.worldbank.org/indicator/IT.NET.USER.ZS → Download → CSV
2. Go to https://data.worldbank.org/indicator/SL.UEM.TOTL.ZS → Download → CSV
3. Place both CSV files in the same folder as this notebook
4. Update the filenames below if needed

In [ ]:
def load_wb(filepath, varname):
    """Load and reshape World Bank CSV into long format."""
    df = pd.read_csv(filepath, skiprows=4)
    df = df.rename(columns={'Country Name': 'country', 'Country Code': 'iso3'})
    year_cols = [c for c in df.columns if c.isdigit()]
    df = df[['country', 'iso3'] + year_cols]
    df = df.melt(id_vars=['country', 'iso3'], value_vars=year_cols,
                 var_name='year', value_name=varname)
    df['year'] = df['year'].astype(int)
    return df

# !! Update filenames if yours are different !!
internet = load_wb('IT.NET.USER.ZS_Data.csv', 'internet')
unemp    = load_wb('SL.UEM.TOTL.ZS_Data.csv', 'unemp')

print(f'Internet data: {internet.shape}')
print(f'Unemployment data: {unemp.shape}')
internet.head(3)

---
## 2. Merge & Clean

In [ ]:
# Merge on country + year
df = pd.merge(internet, unemp, on=['country', 'iso3', 'year'])

# Keep 2000-2022
df = df[(df['year'] >= 2000) & (df['year'] <= 2022)]

# Drop missing
df = df.dropna(subset=['internet', 'unemp'])

# Log internet (for preferred specification)
df['log_internet'] = np.log(df['internet'] + 1)

print(f"Countries: {df['iso3'].nunique()}")
print(f"Years: {df['year'].min()} – {df['year'].max()}")
print(f"Total observations: {len(df)}")
df.head()

---
## 3. Descriptive Statistics

In [ ]:
df[['internet', 'unemp']].describe().round(2)

In [ ]:
# Missing values check
print('Missing values before cleaning:')
print(pd.merge(internet, unemp, on=['country', 'iso3', 'year'])
      [(pd.merge(internet, unemp, on=['country', 'iso3', 'year'])['year'] >= 2000) &
       (pd.merge(internet, unemp, on=['country', 'iso3', 'year'])['year'] <= 2022)]
      [['internet', 'unemp']].isnull().sum())

---
## 4. Main Figure

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# --- Panel A: Scatter plot ---
axes[0].scatter(df['internet'], df['unemp'], alpha=0.3, s=10, color='steelblue')
z = np.polyfit(df['internet'], df['unemp'], 1)
p = np.poly1d(z)
x_line = np.linspace(df['internet'].min(), df['internet'].max(), 100)
axes[0].plot(x_line, p(x_line), color='red', linewidth=2, label='OLS fit')
axes[0].set_xlabel('Internet users (% of population)', fontsize=12)
axes[0].set_ylabel('Unemployment rate (%)', fontsize=12)
axes[0].set_title('Panel A: Internet penetration vs Unemployment', fontsize=13)
axes[0].legend()

# --- Panel B: Global trends over time ---
time_avg = df.groupby('year')[['internet', 'unemp']].mean()
ax2 = axes[1]
ax2.plot(time_avg.index, time_avg['internet'], color='steelblue',
         linewidth=2.5, label='Internet users (%)')
ax2.set_ylabel('Internet users (% of population)', color='steelblue', fontsize=12)
ax2.tick_params(axis='y', labelcolor='steelblue')
ax2_twin = ax2.twinx()
ax2_twin.plot(time_avg.index, time_avg['unemp'], color='red',
              linewidth=2.5, linestyle='--', label='Unemployment (%)')
ax2_twin.set_ylabel('Unemployment rate (%)', color='red', fontsize=12)
ax2_twin.tick_params(axis='y', labelcolor='red')
ax2.set_xlabel('Year', fontsize=12)
ax2.set_title('Panel B: Global trends over time', fontsize=13)
ax2.legend(loc='upper left')
ax2_twin.legend(loc='upper right')

plt.suptitle('Internet Penetration and Unemployment: Global Panel 2000–2022',
             fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig('main_figure.png', dpi=150, bbox_inches='tight')
plt.show()
print('Figure saved as main_figure.png')

---
## 5. Regressions

We estimate four models, progressing from simple OLS to panel fixed effects:

| Model | Specification | Purpose |
|---|---|---|
| 1 | OLS | Baseline correlation |
| 2 | OLS + Year FE | Control for global time trends |
| 3 | Panel FE (country + year) | **Main specification** |
| 4 | Panel FE + log internet | Preferred specification |

In [ ]:
# Model 1: Simple OLS
m1 = smf.ols('unemp ~ internet', data=df).fit()
print('Model 1 — Simple OLS')
print(f"  internet coef: {m1.params['internet']:.4f} (se: {m1.bse['internet']:.4f}), R2: {m1.rsquared:.3f}")

In [ ]:
# Model 2: OLS + Year FE
m2 = smf.ols('unemp ~ internet + C(year)', data=df).fit()
print('Model 2 — OLS + Year Fixed Effects')
print(f"  internet coef: {m2.params['internet']:.4f} (se: {m2.bse['internet']:.4f}), R2: {m2.rsquared:.3f}")

In [ ]:
# Model 3: Panel FE — country + year fixed effects (MAIN SPECIFICATION)
df_panel = df.set_index(['iso3', 'year'])
m3 = PanelOLS.from_formula(
    'unemp ~ internet + EntityEffects + TimeEffects',
    data=df_panel
).fit(cov_type='clustered', cluster_entity=True)
print('Model 3 — Panel FE (Country + Year) — MAIN SPECIFICATION')
print(f"  internet coef: {m3.params['internet']:.4f} (se: {m3.std_errors['internet']:.4f})")
print(m3.summary.tables[1])

In [ ]:
# Model 4: Panel FE + log internet (PREFERRED SPECIFICATION)
df_panel2 = df.set_index(['iso3', 'year'])
m4 = PanelOLS.from_formula(
    'unemp ~ log_internet + EntityEffects + TimeEffects',
    data=df_panel2
).fit(cov_type='clustered', cluster_entity=True)
print('Model 4 — Panel FE + Log Internet — PREFERRED SPECIFICATION')
print(f"  log_internet coef: {m4.params['log_internet']:.4f} (se: {m4.std_errors['log_internet']:.4f})")
print(m4.summary.tables[1])

---
## 6. Results Table

In [ ]:
def stars(pval):
    return '***' if pval < 0.01 else '**' if pval < 0.05 else '*' if pval < 0.1 else ''

results = pd.DataFrame({
    'Model 1 OLS': [
        f"{m1.params['internet']:.4f}{stars(m1.pvalues['internet'])}",
        f"({m1.bse['internet']:.4f})",
        '', '', 'No', 'No',
        str(int(m1.nobs)), f"{m1.rsquared:.3f}"
    ],
    'Model 2 OLS+YearFE': [
        f"{m2.params['internet']:.4f}{stars(m2.pvalues['internet'])}",
        f"({m2.bse['internet']:.4f})",
        '', '', 'No', 'Yes',
        str(int(m2.nobs)), f"{m2.rsquared:.3f}"
    ],
    'Model 3 Panel FE': [
        f"{m3.params['internet']:.4f}{stars(m3.pvalues['internet'])}",
        f"({m3.std_errors['internet']:.4f})",
        '', '', 'Yes', 'Yes',
        str(int(m3.nobs)), f"{m3.rsquared:.3f}"
    ],
    'Model 4 Panel FE log': [
        '',
        '',
        f"{m4.params['log_internet']:.4f}{stars(m4.pvalues['log_internet'])}",
        f"({m4.std_errors['log_internet']:.4f})",
        'Yes', 'Yes',
        str(int(m4.nobs)), f"{m4.rsquared:.3f}"
    ]
}, index=[
    'Internet (%)', '(se)',
    'Log Internet', '(se)',
    'Country FE', 'Year FE',
    'Observations', 'R-squared'
])

print('\n=== MAIN RESULTS TABLE ===')
print(results.to_string())
print('\nNote: * p<0.1, ** p<0.05, *** p<0.01')
print('Clustered standard errors at country level in Models 3-4')

---
## 7. Robustness Check — Drop outliers

In [ ]:
# Drop countries with extreme unemployment (>40%) as robustness check
df_robust = df[df['unemp'] < 40]
df_rob_panel = df_robust.set_index(['iso3', 'year'])

m_robust = PanelOLS.from_formula(
    'unemp ~ internet + EntityEffects + TimeEffects',
    data=df_rob_panel
).fit(cov_type='clustered', cluster_entity=True)

print('Robustness: Drop countries with unemployment > 40%')
print(f"  internet coef: {m_robust.params['internet']:.4f}"
      f" (se: {m_robust.std_errors['internet']:.4f})"
      f", N = {m_robust.nobs}")

---
## 8. Heterogeneity — High vs Low/Middle Income Countries

In [ ]:
high_income = ['USA','GBR','DEU','FRA','JPN','AUS','CAN','KOR','NLD',
               'CHE','SWE','NOR','DNK','FIN','AUT','BEL','NZL','ISR',
               'SGP','HKG','IRL','LUX','ISL','EST','CZE','SVN','SVK',
               'LVA','LTU','HRV','POL','HUN','GRC','PRT','ESP','ITA']

df['income_group'] = df['iso3'].apply(
    lambda x: 'High income' if x in high_income else 'Low/Middle income'
)

print('=== Heterogeneity by Income Group ===')
for group in ['High income', 'Low/Middle income']:
    sub = df[df['income_group'] == group].set_index(['iso3', 'year'])
    try:
        m_het = PanelOLS.from_formula(
            'unemp ~ internet + EntityEffects + TimeEffects',
            data=sub
        ).fit(cov_type='clustered', cluster_entity=True)
        coef  = m_het.params['internet']
        se    = m_het.std_errors['internet']
        pval  = m_het.pvalues['internet']
        s     = stars(pval)
        print(f"  {group}: coef = {coef:.4f}{s} (se = {se:.4f}), N = {m_het.nobs}")
    except Exception as e:
        print(f"  {group}: could not estimate — {e}")

---
## 9. Heterogeneity Figure

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for i, group in enumerate(['High income', 'Low/Middle income']):
    sub = df[df['income_group'] == group]
    axes[i].scatter(sub['internet'], sub['unemp'], alpha=0.3, s=10, color='steelblue')
    z = np.polyfit(sub['internet'], sub['unemp'], 1)
    p = np.poly1d(z)
    x_line = np.linspace(sub['internet'].min(), sub['internet'].max(), 100)
    axes[i].plot(x_line, p(x_line), color='red', linewidth=2)
    axes[i].set_xlabel('Internet users (% of population)', fontsize=11)
    axes[i].set_ylabel('Unemployment rate (%)', fontsize=11)
    axes[i].set_title(f'{group}', fontsize=13)

plt.suptitle('Heterogeneity: Internet vs Unemployment by Income Group',
             fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig('heterogeneity_figure.png', dpi=150, bbox_inches='tight')
plt.show()
print('Figure saved as heterogeneity_figure.png')

---
## 10. Summary & Interpretation

Write your interpretation here based on the results above:

- **Main result (Model 3):** A 1 percentage point increase in internet penetration is associated with a `[FILL IN]` percentage point change in the unemployment rate, controlling for country and year fixed effects.

- **Does it show causal effect?** The panel FE specification controls for all time-invariant country differences and global time trends. However, reverse causality remains possible — richer, lower-unemployment countries may adopt internet faster.

- **Heterogeneity:** The effect is `[larger/smaller]` in high-income countries, suggesting `[FILL IN interpretation]`.

- **Policy conclusion:** `[FILL IN based on your results]`